In [4]:
#!!!!!! THERE IS A MISTAKE IN RUNGE KUTTA FUNCTION !!!!!!

import numpy as np
import copy
import matplotlib.pyplot as plt

In [ ]:
def init_sq_th(alpha, theta, mean_n):

    sq_matrix = np.zeros((4,4), dtype=np.complex128)
    np.fill_diagonal(sq_matrix, np.cosh(alpha))

    sq_matrix[0][2], sq_matrix[2][0] = np.sinh(alpha)*np.cos(theta), np.sinh(alpha)*np.cos(theta)
    sq_matrix[0][3], sq_matrix[3][0] = np.sinh(alpha)*np.sin(theta), np.sinh(alpha)*np.sin(theta)
    sq_matrix[1][2], sq_matrix[2][1] = np.sinh(alpha)*np.sin(theta), np.sinh(alpha)*np.sin(theta)
    sq_matrix[1][3], sq_matrix[3][1] = -np.sinh(alpha)*np.cos(theta), -np.sinh(alpha)*np.cos(theta)
    array = (mean_n + 1/2)*np.matmul(sq_matrix,sq_matrix) 
    return array

def inject_sq_th(cov_M_: np.array, sq_th_M_: np.array):
    cov_M_[:4,:4] = sq_th_M_

def init_cov_full(sq_th_M_: np.array):
    cov_full_ = np.zeros((12,12), dtype=np.complex128)
    np.fill_diagonal(cov_full_, 1/2)
    inject_sq_th(cov_full_, sq_th_M_)
    return cov_full_

def init_A(gamma_, P_, W_in_, J_ij_, eta_):
    A = np.zeros((12,12), dtype=np.complex128)
    #First 1-4x1-4 submatrix
    A[0][0], A[1][1], A[2][2], A[3][3] =  -eta_/(np.sqrt(2)*gamma_)+0j, 0-1j*eta_/(np.sqrt(2)*gamma_), -eta_/(np.sqrt(2)*gamma_)+0j, 0-1j*eta_/(np.sqrt(2)*gamma_)
    #5-12x1-4 submatrix
    A[4][0], A[6][0], A[8][0], A[10][0] = -np.sqrt(2)*W_in_[0]+0j, -np.sqrt(2)*W_in_[1]+0j, -np.sqrt(2)*W_in_[2]+0j, -np.sqrt(2)*W_in_[3]+0j
    A[5][1], A[7][1], A[9][1], A[11][1] = 0-1j*np.sqrt(2)*W_in_[0], 0-1j*np.sqrt(2)*W_in_[1], 0-1j*np.sqrt(2)*W_in_[2], 0-1j*np.sqrt(2)*W_in_[3]
    A[4][2], A[6][2], A[8][2], A[10][2] = -np.sqrt(2)*W_in_[0]+0j, -np.sqrt(2)*W_in_[1]+0j, -np.sqrt(2)*W_in_[2]+0j, -np.sqrt(2)*W_in_[3]+0j
    A[5][3], A[7][3], A[9][3], A[11][3] = 0-1j*np.sqrt(2)*W_in_[0], 0-1j*np.sqrt(2)*W_in_[1], 0-1j*np.sqrt(2)*W_in_[2], 0-1j*np.sqrt(2)*W_in_[3]
    #5-12x5-12 submatrix diagonal
    A[4][4], A[6][6], A[8][8], A[10][10] = -(gamma_ + P_)/np.sqrt(2)+0j, -(gamma_ + P_)/np.sqrt(2)+0j, -(gamma_ + P_)/np.sqrt(2)+0j, -(gamma_ + P_)/np.sqrt(2)+0j
    A[5][5], A[7][7], A[9][9], A[11][11] = 0-1j*(gamma_ + P_)/np.sqrt(2), 0-1j*(gamma_ + P_)/np.sqrt(2), 0-1j*(gamma_ + P_)/np.sqrt(2), 0-1j*(gamma_ + P_)/np.sqrt(2)
    #5-12x5-12 submatrix other values
    A[4][5], A[4][7], A[6][7], A[6][9] = np.sqrt(2)*J_ij_[3]+0j, np.sqrt(2)*J_ij_[0]+0j, np.sqrt(2)*J_ij_[0]+0j, np.sqrt(2)*J_ij_[1]+0j
    A[8][9], A[8][11], A[10][11], A[10][5] = np.sqrt(2)*J_ij_[1]+0j, np.sqrt(2)*J_ij_[2]+0j, np.sqrt(2)*J_ij_[2]+0j, np.sqrt(2)*J_ij_[3]+0j
    A[5][4], A[5][6], A[7][6], A[7][8] = 0-1j*np.sqrt(2)*J_ij_[3], 0-1j*np.sqrt(2)*J_ij_[0], 0-1j*np.sqrt(2)*J_ij_[0], 0-1j*np.sqrt(2)*J_ij_[1]
    A[9][8], A[9][10], A[11][10], A[11][4] = 0-1j*np.sqrt(2)*J_ij_[1], 0-1j*np.sqrt(2)*J_ij_[2], 0-1j*np.sqrt(2)*J_ij_[2], 0-1j*np.sqrt(2)*J_ij_[3]

    return A

def rungekutta(first_: np.complex128, second_: np.complex128, time_start_, current_time_, time_step_, tau_):

    def func_f(next_time_):
        if time_start_ < next_time_ < tau_:
            return first_
        #HERE IS A MISTAKE, THE RANGES ARE OPEN NOT CLOSED x1 < y < x_2
        elif time_start_ + tau_ <= next_time_ < time_start_ + 2*tau_:
            return second_
        else:
            return 0
                
    k_1 = func_f(current_time_)
    k_2 = func_f(current_time_ + time_step_/2)
    k_3 = func_f(current_time_ + time_step_/2)
    k_4 = func_f(current_time_ + time_step_)

    return (time_step_/6)*(k_1+2*k_2+2*k_3+k_4)

def time_evolve(cov_M_, A_, time_start_, time_end_, time_step_, tau_):
    #Initialize first mode
    first_mode = copy.copy(A_)
    first_mode[:,[2,3]] = 0
    #Initialize second mode
    second_mode = copy.copy(A_)
    second_mode[:,[0,1]] = 0
    #First mode interaction
    cov_first = np.matmul(first_mode, cov_M_) + np.matmul(cov_M_, np.transpose(first_mode))
    #Second mode interaction
    cov_second = np.matmul(second_mode, cov_M_) + np.matmul(cov_M_, np.transpose(second_mode))

    current_time_ = time_start_

    while current_time_ <= time_end_:

        for (idx, val1), (_, val2) in zip(np.ndenumerate(cov_first), np.ndenumerate(cov_second)):
            cov_M_[idx[0]][idx[1]] = cov_M_[idx[0]][idx[1]] + rungekutta(val1, val2, time_start_, current_time_, time_step_, tau_)
            
        current_time_ += time_step_

def get_occupation_nums(cov_M_: np.array, reservoir_size_: int):
    occupation_numbers_ = np.empty([1,reservoir_size_], dtype=np.complex128)

    for i in range(reservoir_size_):
        occupation_numbers_[i] = 1/2*(cov_M_[4+2*i][4+2*i] + cov_M_[5+2*i][5+2*i] - 1)
    
    return occupation_numbers_
    
def update_W_out(Y_true_: np.array, occupation_numbers_: np.array, W_out_old_: np.array):
    




In [ ]:
#Initialize variables
number_of_inputs = 10

theta = np.random.uniform(0,2*np.pi,(number_of_inputs,))
s = np.random.uniform(0.8,0.95,(number_of_inputs,))
phi = np.random.uniform(0.5-np.pi/10, 0.5+np.pi/10, (number_of_inputs,))
abs_alpha = np.array([x*np.sin(y) for x, y in zip(s,phi)])
mean_n = np.array([x*x*np.cos(y)*np.cos(y) for x, y in zip(s,phi)])

gamma = 1
P = 0.1*gamma
tau = 1/gamma
J_ij = np.random.uniform(-gamma, gamma, (4,))
W_in = np.random.uniform(0, gamma, (4,))
eta = np.sum(W_in**2)

#Initialize initial covariance matrix of system
cov_sq_th = init_sq_th(abs_alpha[0], theta[0], mean_n[0])
cov_full = init_cov_full(cov_sq_th)
matrix_A = init_A(gamma, P, W_in, J_ij, eta)

time_start = 0
time_end = 1
time_step = 1/3

W_out = np
Y_out = np.empty([1,number_of_inputs], dtype=np.complex128)

In [ ]:
jeps = np.array([[1,2,3],[4,5,6],[7,8,9]])

joo = np.array()


mean_n = np.array([x*x*np.cos(y)*np.cos(y) for x, y in zip(s,phi)])

In [7]:

time_evolve(cov_full, matrix_A, time_start, time_end, time_step, tau)

for input_node in range(1, number_of_inputs):
    inject_sq_th(cov_full, init_sq_th(abs_alpha[input_node], theta[input_node], mean_n[input_node]))
    time_evolve(cov_full, matrix_A, time_start, time_end, time_step, tau)